# 04 — Workflow model and metrics

**Class 7 deliverable.** Represent entities, events and outcomes; build the model;
calculate metrics linked to the KPI.

In [1]:
import sys, json, duckdb, pandas as pd
sys.path.insert(0, '../src')
from pipeline.config import load_config
cfg = load_config('../config/pipeline.yaml')
con = duckdb.connect()
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 50)
RAW, MARTS, STG = '../data/raw', '../data/marts', '../data/staging'

## 1. The workflow, and what the data withholds

```
[request] --> [pickup] --> [in transit] --> [dropoff] --> [paid]
    |                           |
    |                           +-- the ONLY observable interval
    +-- NOT RECORDED: unmet demand is invisible
```

The interval `pickup_ts → dropoff_ts` plus `trip_distance_mi` is the entire basis
of the KPI: it yields an average speed, the closest observable proxy for *"was this
trip impeded?"*. The model is built around that boundary rather than pretending
past it.

## 2. The star schema

`fct_trip` (event) joined to `dim_zone` (twice — pickup and dropoff) and
`fct_weather_hour`. Full ER diagram in
[`docs/data_model.md`](../docs/data_model.md).

In [2]:
t = con.sql(f"SELECT * FROM read_parquet('{STG}/fct_trip/month=2024-01/*.parquet') LIMIT 2").df()
print(f"fct_trip has {len(t.columns)} columns. The ones that carry the analysis:\n")
display(t[['fleet', 'pickup_ts', 'dropoff_ts', 'trip_distance_mi', 'duration_min',
           'speed_mph', 'pu_borough', 'is_airport_trip', 'is_wet_hour',
           'is_valid_core', 'is_congested']])

fct_trip has 52 columns. The ones that carry the analysis:



,fleet,pickup_ts,dropoff_ts,trip_distance_mi,duration_min,speed_mph,pu_borough,is_airport_trip,is_wet_hour,is_valid_core,is_congested
0,yellow,2024-01-14 12:13:16,2024-01-14 12:18:11,0.81,4.916667,9.884746,Manhattan,False,False,True,False
1,yellow,2024-01-14 12:20:37,2024-01-14 12:25:04,1.05,4.450000,14.157303,Manhattan,False,False,True,False


## 3. Derived fields — three deliberate choices

In [3]:
print(con.sql(f'''
    SELECT
      count(*) AS total,
      count(*) FILTER (WHERE speed_mph IS NULL)     AS speed_null,
      count(*) FILTER (WHERE is_congested IS NULL)  AS congested_null,
      count(*) FILTER (WHERE NOT is_valid_core)     AS invalid_core
    FROM read_parquet('{STG}/fct_trip/month=2024-01/*.parquet')
''').df().T.rename(columns={0: 'count'}))

                  count
total           3021175
speed_null          946
congested_null    77255
invalid_core      77255


`is_congested` is **NULL**, not FALSE, outside the valid population — and the count
matches `invalid_core` exactly. That is deliberate: if invalid trips defaulted to
FALSE, `sum(CASE WHEN is_congested ...)` would quietly count them as
*not congested* and dilute the KPI. NULL forces every consumer to filter to
`is_valid_core` explicitly.

Likewise `speed_mph` is NULL when duration ≤ 0, so a bad timestamp can never
become a fake speed.

## 4. Join integrity — the silent fan-out check

In [4]:
staged = con.sql(f"SELECT count(*) FROM read_parquet('{STG}/fct_trip/month=2024-01/*.parquet')").fetchone()[0]
print(f"rows after all three joins: {staged:,}")
print(con.sql(f'''
    SELECT round(100.0*count(*) FILTER (WHERE pu_borough IS NOT NULL)/count(*), 3) AS zone_join_pct,
           round(100.0*count(*) FILTER (WHERE temp_c IS NOT NULL)/count(*), 3)     AS weather_join_pct
    FROM read_parquet('{STG}/fct_trip/month=2024-01/*.parquet') WHERE is_valid_core
''').df())

rows after all three joins: 3,021,175
   zone_join_pct  weather_join_pct
0          100.0             100.0


A `LEFT JOIN` that *changes the row count* means a duplicated dimension key — the
classic silent fan-out bug that inflates every downstream metric. The pipeline
asserts `staged == joined` and aborts if not (check C7), and separately requires
≥99.9% join coverage (C8).

## 5. The five metrics

In [5]:
import os
for m in sorted(os.listdir(MARTS)):
    n = con.sql(f"SELECT count(*) FROM read_parquet('{MARTS}/{m}/**/*.parquet')").fetchone()[0]
    print(f"{m:<22} {n:>6} rows across 6 months")

m_airport                  24 rows across 6 months
m_congested_rate         2598 rows across 6 months
m_data_quality             96 rows across 6 months
m_speed_duration         1363 rows across 6 months
m_weather_effect           48 rows across 6 months


| Metric | Question | Population |
|---|---|---|
| `m_congested_rate` | **KPI** — where/when is service slow? | `is_valid_core AND is_valid_zone` |
| `m_speed_duration` | How slow, and for how long? | `is_valid_core AND is_valid_zone` |
| `m_airport` | Are airport runs different? | `is_valid_core` |
| `m_weather_effect` | Weather-driven or structural? | `is_valid_core` |
| `m_data_quality` | Can we trust the denominator? | all rows |

Definitions live in [`sql/metrics.sql`](../sql/metrics.sql).

## 6. Reconciliation — the KPI computed two ways must agree

In [6]:
direct = con.sql(f'''
    SELECT round(100.0*count(*) FILTER (WHERE is_congested)/count(*), 4)
    FROM read_parquet('{STG}/fct_trip/month=2024-01/*.parquet')
    WHERE is_valid_core AND is_valid_zone
''').fetchone()[0]
mart = con.sql(f'''
    SELECT round(100.0*sum(congested_trips)/sum(trips), 4)
    FROM read_parquet('{MARTS}/m_congested_rate/month=2024-01/*.parquet')
''').fetchone()[0]
print(f"from fct_trip     : {direct}%")
print(f"from published mart: {mart}%")
print(f"agree: {abs(direct - mart) < 0.01}")

from fct_trip     : 31.4706%
from published mart: 31.4706%
agree: True


This runs on every month as check **C10**. Together with **C9** (valid + excluded
= total), it means no trip can vanish between raw extract and published metric
without the run failing.

## 7. Sensitivity — how much does the 8 mph choice matter?

In [7]:
sens = con.sql(f'''
    SELECT round(100.0*avg(CASE WHEN speed_mph < 6  THEN 1.0 ELSE 0 END), 2) AS "6 mph",
           round(100.0*avg(CASE WHEN speed_mph < 7  THEN 1.0 ELSE 0 END), 2) AS "7 mph",
           round(100.0*avg(CASE WHEN speed_mph < 8  THEN 1.0 ELSE 0 END), 2) AS "8 mph",
           round(100.0*avg(CASE WHEN speed_mph < 9  THEN 1.0 ELSE 0 END), 2) AS "9 mph",
           round(100.0*avg(CASE WHEN speed_mph < 10 THEN 1.0 ELSE 0 END), 2) AS "10 mph"
    FROM read_parquet('{STG}/fct_trip/**/*.parquet') WHERE is_valid_core
''').df()
display(sens.T.rename(columns={0: 'congested %'}))

,congested %
6 mph,14.74
7 mph,23.84
8 mph,34.31
9 mph,45.04
10 mph,55.00


In [8]:
# The level moves a lot. Does the RANKING - what ops actually acts on - move?
for th in (6, 8, 10):
    r = con.sql(f'''
        SELECT pu_borough, round(100.0*avg(CASE WHEN speed_mph < {th} THEN 1.0 ELSE 0 END), 1) k
        FROM read_parquet('{STG}/fct_trip/**/*.parquet')
        WHERE is_valid_core AND is_valid_zone
          AND pu_borough IN ('Manhattan','Brooklyn','Queens','Bronx')
        GROUP BY 1 ORDER BY k DESC
    ''').fetchall()
    print(f"{th:>2} mph:  " + "  >  ".join(f"{b} ({k})" for b, k in r))

 6 mph:  Manhattan (16.2)  >  Brooklyn (12.0)  >  Bronx (9.7)  >  Queens (1.1)
 8 mph:  Manhattan (37.7)  >  Brooklyn (28.1)  >  Bronx (21.2)  >  Queens (2.8)
10 mph:  Manhattan (60.3)  >  Brooklyn (46.8)  >  Bronx (36.4)  >  Queens (6.0)


**This is the central caveat.** The KPI's absolute level swings from 14.7% to
55.0% depending on the threshold — *"34% of trips are congested" is a fact about
an 8 mph line, not about New York.*

But the borough ranking is **identical at every threshold tested**. So the output
is dependable for the decision it actually supports — ranking zones and hours,
and tracking change month over month — and should not be quoted as an absolute
measure. Recorded as limitation **L1**.